In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
base = 'aaus7_class_'
file_format = '.csv'
tel = pd.read_csv(base+'timestamped_telemetry'+file_format,parse_dates=['observed_at'])
#tel.info()
ops = pd.read_csv(base+'operations_log'+file_format,parse_dates=['logged_at'])
#ops.info()
bal = pd.read_csv(base+'daily_water_balance'+file_format,parse_dates=['report_date']) 
#bal.info()
inc = pd.read_csv(base+'service_incidents'+file_format,parse_dates=['event_time','closed_at'])
#inc.info()
asset = pd.read_csv(base+'asset_reference'+file_format,parse_dates=['installation_year'])
#asset.info()
kpi = pd.read_csv(base+'monthly_kpis_wide'+file_format,parse_dates=['month'])
kpi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 7 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   month                      36 non-null     datetime64[ns]
 1   zone                       36 non-null     object        
 2   supply_reliability_pct     36 non-null     float64       
 3   billing_collection_pct     36 non-null     float64       
 4   quality_compliance_pct     36 non-null     float64       
 5   customer_satisfaction_pct  36 non-null     float64       
 6   report_status              36 non-null     object        
dtypes: datetime64[ns](1), float64(4), object(2)
memory usage: 2.1+ KB


In [24]:
left = pd.DataFrame({
'day':[1,3],
'supplied':[40,52]})
right = pd.DataFrame({
'day':[2,3],
'target':[45,50]})
ordered = pd.merge_ordered(
left, right, on='day')
print(ordered.to_string(index=False))

 day  supplied  target
   1      40.0     NaN
   2       NaN    45.0
   3      52.0    50.0


In [25]:
filled = pd.merge_ordered(
left, right,
on='day',
fill_method='ffill'
)
print(filled.to_string(index=False))

 day  supplied  target
   1        40     NaN
   2        40    45.0
   3        52    50.0


In [28]:
tl = tel.sort_values('observed_at')
op = ops.sort_values('logged_at')
print('sensor sorted:', 
tl.observed_at.is_monotonic_increasing)
print('log sorted:', 
op.logged_at.is_monotonic_increasing)
print('zones:', tl.zone.nunique())
tl
op

sensor sorted: True
log sorted: True
zones: 6


,log_id,logged_at,zone,pump_mode,active_units,power_event,staff_shift,supervisor_team
8,OP-00009,2026-01-05 05:01:00,Gullele,Conservation,3,No,Early,Ops-C
4,OP-00005,2026-01-05 05:06:00,Lideta,Normal,5,No,Early,Ops-A
0,OP-00001,2026-01-05 05:11:00,Akaki,Conservation,3,No,Early,Ops-B
6,OP-00007,2026-01-05 05:17:00,Merkato,Normal,2,No,Early,Ops-B
2,OP-00003,2026-01-05 05:22:00,Kirkos,Booster,4,No,Early,Ops-C
...,...,...,...,...,...,...,...,...
986,OP-00987,2026-04-04 17:08:00,Bole,Booster,3,No,Late,Ops-A
996,OP-00997,2026-04-04 17:09:00,Gullele,Normal,4,No,Late,Ops-C
992,OP-00993,2026-04-04 17:14:00,Lideta,Conservation,2,No,Late,Ops-A
988,OP-00989,2026-04-04 17:19:00,Akaki,Normal,4,No,Late,Ops-B


In [30]:
aligned = pd.merge_asof(
tl, op,
left_on='observed_at',
right_on='logged_at', by='zone',
direction='backward',
tolerance=pd.Timedelta('18h'))
print('left rows:', len(tl), 'joined:', 
len(aligned))
print('matched:', 
int(aligned.log_id.notna().sum()))
aligned

left rows: 1045 joined: 1045
matched: 1016


,reading_id,observed_at,zone,station_id,sensor_vendor,shift,pressure_bar,flow_lps,chlorine_mg_l,battery_pct,quality_flag,log_id,logged_at,pump_mode,active_units,power_event,staff_shift,supervisor_team
0,TR-00008,2026-01-05 08:05:00,Merkato,ST-ME-01,HydroSense,Day,2.08,81.8,0.410,100,Valid,OP-00007,2026-01-05 05:17:00,Normal,2.0,No,Early,Ops-B
1,TR-00002,2026-01-05 08:07:00,Akaki,ST-AK-01,HydroSense,Day,1.69,66.2,0.290,79,Valid,OP-00001,2026-01-05 05:11:00,Conservation,3.0,No,Early,Ops-B
2,TR-00010,2026-01-05 08:12:00,Gullele,ST-GU-01,Civic Metering,Day,2.21,87.0,0.450,78,Valid,OP-00009,2026-01-05 05:01:00,Conservation,3.0,No,Early,Ops-C
3,TR-00004,2026-01-05 08:14:00,Kirkos,ST-KI-01,Civic Metering,Day,1.82,71.4,0.330,86,Valid,OP-00003,2026-01-05 05:22:00,Booster,4.0,No,Early,Ops-C
4,TR-00006,2026-01-05 08:21:00,Lideta,ST-LI-01,Acme Instruments,Day,1.95,76.6,0.370,93,Valid,OP-00005,2026-01-05 05:06:00,Normal,5.0,No,Early,Ops-A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1040,TR-01035,2026-04-04 20:04:00,Bole,ST-BS-03,Acme Instruments,Night,2.12,84.4,0.268,83,Valid,OP-00987,2026-04-04 17:08:00,Booster,3.0,No,Late,Ops-A
1041,TR-01043,2026-04-04 20:09:00,Merkato,ST-ME-03,HydroSense,Night,2.64,105.2,0.428,82,Valid,OP-00995,2026-04-04 17:25:00,Booster,3.0,No,Late,Ops-B
1042,TR-01037,2026-04-04 20:11:00,Akaki,ST-AK-03,HydroSense,Night,2.25,89.6,0.308,90,Valid,OP-00989,2026-04-04 17:19:00,Normal,4.0,No,Late,Ops-B
1043,TR-01045,2026-04-04 20:16:00,Gullele,ST-GU-03,Civic Metering,Night,2.77,110.4,0.468,89,Valid,OP-00997,2026-04-04 17:09:00,Normal,4.0,No,Late,Ops-C
